# Step 06 â€” LLM Building Classification

**Input:** `data/output/05_condensed_buildings_with_pois.gpkg`  
**Output:** Predictions saved incrementally to `data/output/llm_predictions/`
- `predictions_checkpoint.parquet` â€” all successful predictions so far
- `prediction_errors.parquet` â€” rows that failed validation (picked up by notebook 07)

**Features (from linux branch):**
- Checkpoint/resume: safe to interrupt and restart â€” already-processed rows are skipped
- ThreadPoolExecutor: parallel API calls (`LLM_MAX_WORKERS` from config)
- Per-chunk flush to parquet: memory-efficient for large datasets

**LLM outputs per building:**
- `mid_labels` â€” activity labels (work, school, retail_daily, â€¦)
- `bosserhof_class` â€” building-use type for capacity estimation
- `interpreted_type` â€” plain-English description
- `short_reason` â€” LLM reasoning

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import json
import os
import re
import time
import requests
import pandas as pd
import geopandas as gpd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

load_dotenv(dotenv_path=ROOT / '.env')
TU_TOKEN = os.getenv('TU_KI_TOOLBOX_TOKEN')
if not TU_TOKEN:
    raise RuntimeError('Missing TU_KI_TOOLBOX_TOKEN â€” copy .env.example to .env and add your token')

LLM_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
print('Config loaded. LLM model:', LLM_MODEL)

## System prompt

In [ ]:
SYSTEM_PROMPT = """
You are a building activity interpreter and classifier.

Your task is to classify BUILDINGS using structured input divided into TWO PARTS:
1) precise_known_info â€” high-confidence OSM-derived signals such as names, amenity, shop, office, tourism, healthcare, leisure, etc.
2) general_building_context â€” broader contextual signals such as ALKIS landuse, OSM building type, OSM landuse, auxiliary tags.

You must produce TWO outputs:
1) Activity labels (mid_labels) â€” what activities take place inside the building
2) Bosserhof class â€” the dominant functional building-use class for capacity / volume estimation

CRITICAL SCOPE RULE
Only classify ENTERABLE BUILDINGS or building-like places people actually use as destinations.
If the described place is not a building, not enterable, or only an outdoor / infrastructure / passive object:
- "mid_labels": []
- "bosserhof_class": null

ALLOWED ACTIVITY LABELS (mid_labels):
work, university, school, childcare, retail_daily, retail_non_daily, leisure, sports, errands, meetup, lessons, business

ALLOWED BOSSERHOF HEADLINE CATEGORIES:
1) Transport
2) Yards, depots, storage areas, construction yards
3) Industrial operations / Production  (subcategories: highly productive industries / machine / material or space intensive | others)
4) Crafts and trades  (subcategories: craft businesses | craft courtyards)
5) Services  (subcategories: normal office | open-plan office | business-oriented services | customer-oriented services | hotels | hotels with conference areas | restaurants / gastronomy | suppliers for car dealerships | vehicle / electrical repair | customer service | car dealerships)
6) Retail  (subcategories: wholesale | retail (small-scale) | discount stores | DIY stores | furniture stores | hypermarkets / superstores | shopping centers | self-service department stores | department stores | factory outlet centers)
7) Public facilities  (subcategories: schools | universities | research institutes | kindergartens | hospitals | nursing homes)
8) Facilities for culture, leisure and sports  (subcategories: entertainment, culture | large cinemas | musical theatres | large discos, fun / leisure pools | arenas, large events | theme parks | fitness / wellness)

Use a subcategory string when confident; fall back to headline category when not. Use null only if the building truly does not fit any category.

OUTPUT FORMAT (STRICT JSON ONLY):
{
  "interpreted_type": "<plain-English description>",
  "mid_labels": ["<zero or more labels from the allowed list>"],
  "bosserhof_class": "<one Bosserhof class or null>",
  "reason": "<max 400 words explaining both classifications>"
}
""".strip()

## Helper functions

In [ ]:
from llm_utils import is_missing, format_value, row_to_llm_input, extract_first_json
from llm_utils import validate as _validate

def validate(obj):
    _validate(obj, TARGET_MID_LABELS)


## Load data and build LLM input sentences

In [ ]:
pois = gpd.read_file(CONDENSED_BUILDINGS_FILE)
print(f'Loaded {len(pois):,} buildings')

pois['sentence'] = pois.apply(row_to_llm_input, axis=1)
pois.head(2)

## Run LLM classification with checkpoint/resume

**Safe to restart**: already-processed `gml_id`s are loaded from the checkpoint and skipped.

In [ ]:
# Load checkpoint if it exists
done_ids = set()
if LLM_CHECKPOINT_FILE.exists():
    checkpoint_df = pd.read_parquet(LLM_CHECKPOINT_FILE)
    done_ids = set(checkpoint_df['gml_id'].astype(str))
    print(f'Resuming from checkpoint: {len(done_ids):,} already done')

todo = pois[~pois['gml_id'].astype(str).isin(done_ids)].copy()
print(f'Remaining to classify: {len(todo):,}')

def process_chunk(chunk_df):
    results = []
    with ThreadPoolExecutor(max_workers=LLM_MAX_WORKERS) as executor:
        futures = {
            executor.submit(predict_row, row['gml_id'], row['sentence']): i
            for i, row in chunk_df.iterrows()
        }
        for future in as_completed(futures):
            results.append(future.result())
    return results

def append_parquet(path, new_rows):
    new_df = pd.DataFrame(new_rows)
    if path.exists():
        existing = pd.read_parquet(path)
        combined = pd.concat([existing, new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset='gml_id', keep='last')
    else:
        combined = new_df
    combined.to_parquet(path, index=False)

n_chunks = max(1, len(todo) // LLM_CHUNK_SIZE)
chunks   = [todo.iloc[i * LLM_CHUNK_SIZE:(i + 1) * LLM_CHUNK_SIZE] for i in range(n_chunks)]
if len(todo) % LLM_CHUNK_SIZE:
    chunks.append(todo.iloc[n_chunks * LLM_CHUNK_SIZE:])

total_done = len(done_ids)
for i, chunk in enumerate(chunks):
    results = process_chunk(chunk)

    good   = [r for r in results if r['error'] is None]
    errors = [r for r in results if r['error'] is not None]

    if good:   append_parquet(LLM_CHECKPOINT_FILE, good)
    if errors: append_parquet(LLM_ERRORS_FILE, errors)

    total_done += len(results)
    print(f'Chunk {i+1}/{len(chunks)} | done={total_done}/{len(pois)} | errors={len(errors)}')

print('Classification complete.')